# mbpp-tr LoRA fine-tune (Qwen3-1.7B)

`firatmio/mbpp-tr` ile `Qwen/Qwen3-1.7B` üzerinde LoRA eğitimi ve **çalıştırılarak** ölçülen pass@1 değerlendirmesi.

Akış: harness doğrulama → baseline eval → eğitim → LoRA eval → karşılaştırma → Hub'a yükleme.

**Runtime:** `Runtime > Change runtime type > T4 GPU`

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/firatmio/mbpp-tr-finetune.git
%cd mbpp-tr-finetune
!pip install -q -U -r requirements.txt

## 1. Harness doğrulama
Referans çözümler kendi testlerini geçmiyorsa, model sonuçları da güvenilir değildir. Beklenen: `257/257`.

In [ ]:
!python scripts/check_references.py --config sanitized --split test

## 2. Baseline (fine-tune öncesi)
Aynı prompt, greedy decoding. T4'te ~10-20 dk sürebilir.

In [ ]:
!python scripts/evaluate.py --out outputs/eval_base

## 3. LoRA eğitimi
Varsayılanlar: r=16, alpha=32, dropout=0.05, lr=2e-4, 3 epoch, efektif batch 16, cosine. Her epoch sonunda validation loss; en iyisi saklanır.

In [ ]:
!python scripts/train_lora.py --output_dir outputs/lora

## 4. LoRA sonrası değerlendirme

In [ ]:
!python scripts/evaluate.py --adapter outputs/lora/final --out outputs/eval_lora

## 5. Karşılaştırma

In [ ]:
import json
rows = {}
for name in ["base", "lora"]:
    with open(f"outputs/eval_{name}/summary.json") as f:
        rows[name] = json.load(f)
for name, s in rows.items():
    print(f"{name:5s} pass@1 = {s['pass@1']:.4f}  ({s['passed']}/{s['n']})  {s['status_counts']}")

# Hangi görevler kazanıldı / kaybedildi?
def load(name):
    with open(f"outputs/eval_{name}/samples.jsonl", encoding="utf-8") as f:
        return {r["task_id"]: r for r in map(json.loads, f)}
base, lora = load("base"), load("lora")
gained = [t for t in base if lora[t]["passed"] and not base[t]["passed"]]
lost = [t for t in base if base[t]["passed"] and not lora[t]["passed"]]
print(f"kazanilan: {len(gained)}  kaybedilen: {len(lost)}")
print("kaybedilenler:", lost)

## 6. Çıktıları sakla ve Hub'a yükle
Colab oturumu kapanınca `outputs/` silinir. Önce Drive'a yedekle, sonra adapter'ı Hub'a yükle (HF token'ı `write` yetkili olmalı).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/mbpp-tr-finetune
!cp -r outputs/eval_base outputs/eval_lora outputs/lora/final outputs/lora/train_summary.json /content/drive/MyDrive/mbpp-tr-finetune/

In [ ]:
from huggingface_hub import login
login()  # token'ı etkileşimli gir; notebook'a yazma

In [ ]:
from huggingface_hub import HfApi

REPO_ID = "firatmio/qwen3-1.7b-mbpp-tr-lora"  # model card hazır olmadan private tutmak mantıklı
api = HfApi()
api.create_repo(REPO_ID, repo_type="model", private=True, exist_ok=True)
api.upload_folder(repo_id=REPO_ID, folder_path="outputs/lora/final", commit_message="LoRA adapter")
for name in ["base", "lora"]:
    api.upload_folder(repo_id=REPO_ID, folder_path=f"outputs/eval_{name}", path_in_repo=f"eval/{name}")
api.upload_file(repo_id=REPO_ID, path_or_fileobj="outputs/lora/train_summary.json", path_in_repo="train_summary.json")